**LLM's Fundacionales para análisis de sentimiento**

Se testean 3 LLMs fundacionales (Robertuito,  bert multilingual, beto) con frases cualquiera que simulan comentarios en redes sociales, para evaluar la lógica de cada modelo.

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 57.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
# Modelo 01:  robertuito-sentiment-analysis
# Se utiliza un frase cualquiera en la variable 'texto' para evaluar el sentimiento que asigna
# El sentimiento se asigna de acuerdo al score de cada etiqueta "POS","NEU","NEG".

from transformers import pipeline
clf = pipeline("text-classification", model="pysentimiento/robertuito-sentiment-analysis", top_k=None)
texto = "si toca con ese banco, da lo mismo"
scores = {s["label"]: s["score"] for s in clf(texto)[0]}
sent = "Positivo" if scores["POS"] > scores["NEG"] else "Neutral" if scores["NEU"] > scores["NEG"] else "Negativo"
print(sent)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

neutral


In [ ]:
# Se muestra el score de cada categoría.

clf(texto)[0]

[{'label': 'NEU', 'score': 0.7703526020050049},
 {'label': 'NEG', 'score': 0.1483726054430008},
 {'label': 'POS', 'score': 0.08127482235431671}]

In [ ]:
# Modelo 02: bert-base-multilingual-uncased-sentiment
# El modelo evalua varios comentarios simulados almacenados en una lista,
# a los cuales asigna un score representado en cantidad de estrellas.

from transformers import pipeline

# Cargar el modelo preentrenado de Hugging Face
modelo = "nlptown/bert-base-multilingual-uncased-sentiment"
clasificador = pipeline("sentiment-analysis", model=modelo)

# Lista de ejemplos (pueden ser textos de redes sociales)
textos = [
    "El servicio fue excelente, muy rápido y amable.",
    "Pésimo servicio",
    "Está bien, pero podría ser mejor.",
]

# Analizar sentimiento
resultados = clasificador(textos)

# Mostrar resultados
for texto, resultado in zip(textos, resultados):
    label = resultado["label"]
    score = round(resultado["score"], 3)
    # El modelo devuelve estrellas: 1 = muy negativo, 5 = muy positivo
    print(f"Texto: {texto}")
    print(f"Predicción: {label} ({score})\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Texto: El servicio fue excelente, muy rápido y amable.
Predicción: 5 stars (0.761)

Texto: Pésimo servicio
Predicción: 1 star (0.896)

Texto: Está bien, pero podría ser mejor.
Predicción: 3 stars (0.69)



In [ ]:
# Se muestra las estrellas y el score de cada categoría.

clasificador(textos)

[{'label': '5 stars', 'score': 0.7609930038452148},
 {'label': '4 stars', 'score': 0.365121066570282},
 {'label': '3 stars', 'score': 0.6898822784423828}]

In [ ]:
# Modelo 03: beto-sentiment-analysis (sobre TASS 2020, ~5k tuits)
# El propio autor indica que está deprecado a favor de RoBERTuito.
# El modelo evalua varios comentarios simulados almacenados en una lista
# y clasifica el comentario en POS o NEG.

from transformers import pipeline

modelo = "finiteautomata/beto-sentiment-analysis"
clf = pipeline("text-classification", model=modelo, top_k = None, truncation=True)

textos = [
    "El producto superó mis expectativas, excelente calidad.",
    "El servicio al cliente fue pésimo y tardaron demasiado.",
    "Regular."
]

def binarizar(scores, margin=0):
    s = {x["label"]: x["score"] for x in scores}
    # Comparo POS vs NEG; ignoro NEU salvo desempate por margen
    if abs(s["POS"] - s["NEG"]) <= margin:
        etiqueta = "positivo" if s["POS"] >= s["NEG"] else "negativo"
        return etiqueta, True
    return ("positivo" if s["POS"] > s["NEG"] else "negativo"), False

for t in textos:
    resultado = clf(t)[0]  # lista de dicts con NEG/NEU/POS
    etiqueta, tie = binarizar(resultado, margin=0)
    print(f"\nTexto: {t}")
    print({r["label"]: round(r["score"],3) for r in resultado})
    print(f"Sentimiento (binario): {etiqueta}{' (desempate)' if tie else ''}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: finiteautomata/beto-sentiment-analysis
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Texto: El producto superó mis expectativas, excelente calidad.
{'POS': 0.998, 'NEG': 0.001, 'NEU': 0.0}
Sentimiento (binario): positivo

Texto: El servicio al cliente fue pésimo y tardaron demasiado.
{'NEG': 0.999, 'POS': 0.0, 'NEU': 0.0}
Sentimiento (binario): negativo

Texto: Regular.
{'NEU': 0.999, 'POS': 0.001, 'NEG': 0.0}
Sentimiento (binario): positivo
